# gpu_layers Test — All Providers

Verifies that `gpu_layers` actually changes VRAM usage.

- **llama_cpp**: `gpu_layers=0` → CPU only, `gpu_layers=99` → full GPU. Expects large VRAM delta.
- **ollama / lm_studio / unsloth**: GPU is auto-managed. We load once and record VRAM to confirm the model lands on GPU. `gpu_layers` is not a supported runtime parameter for these providers.

In [ ]:
import subprocess
import time
from pathlib import Path

from unified_local_llm_server import LLMProviderPool
from unified_local_llm_server.provider_registry import ProviderRegistry

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
registry = ProviderRegistry.load(ROOT / "providers.example.yaml")
server = LLMProviderPool(provider_registry=registry)

PROMPT = "Reply with exactly one word: GPU"
OPTIONS = {"num_predict": 8}

def vram_used_mib() -> int:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            text=True, timeout=5,
        )
        return sum(int(x) for x in out.strip().split("\n") if x.strip())
    except Exception:
        return 0

## Provider Status

In [ ]:
statuses = {}
for name in server.get_providers():
    statuses[name] = await server.check_provider(name)
    ok = statuses[name]["ok"]
    print(f"  {'OK' if ok else '--'} {name:<12} {statuses[name]['server_url']}")

## Preferred Models

In [ ]:
PREFERRED_MODELS = {
    "ollama":    "gpt-oss:20b",
    "lm_studio": "openai/gpt-oss-20b",
    "unsloth":   "unsloth/gpt-oss-20b-GGUF@UD-Q4_K_XL",
    "llama_cpp": "gpt-oss-20b-MXFP4",
}

## llama_cpp: gpu_layers=0 vs gpu_layers=99

Expect near-zero VRAM delta with `gpu_layers=0` and large delta (~12GB) with `gpu_layers=99`.

In [ ]:
provider = "llama_cpp"
model = PREFERRED_MODELS[provider]
messages = [{"role": "user", "content": PROMPT}]
llama_results = []

if not statuses[provider]["ok"]:
    print(f"{provider} SKIP — not reachable")
else:
    for gpu_layers in [0, 99]:
        label = f"gpu_layers={gpu_layers}"
        print(f"\n--- {provider} / {model} / {label} ---")

        try:
            server.unload_all_models(provider)
            time.sleep(2)
        except Exception:
            pass

        vram_before = vram_used_mib()

        try:
            llm = server.load_model(provider, model, gpu_layers=gpu_layers)
            t0 = time.perf_counter()
            result, usage = await llm.call(
                return_usage=True,
                messages=messages,
                temperature=0.0,
                options=OPTIONS,
            )
            elapsed = time.perf_counter() - t0
        except Exception as exc:
            print(f"  ERROR: {exc}")
            llama_results.append({"label": label, "gpu_layers": gpu_layers, "error": str(exc)})
            continue

        vram_after = vram_used_mib()
        vram_delta = vram_after - vram_before
        gen_tok = usage.get("completion_tokens", 0)
        gen_tps = round(gen_tok / elapsed, 1) if elapsed > 0 else 0

        print(f"  VRAM before : {vram_before} MiB")
        print(f"  VRAM after  : {vram_after} MiB")
        print(f"  VRAM delta  : {vram_delta} MiB")
        print(f"  Speed       : {gen_tps} tok/s")
        print(f"  Reply       : {result!r}")

        llama_results.append({
            "label":      label,
            "gpu_layers": gpu_layers,
            "vram_before_mib": vram_before,
            "vram_after_mib":  vram_after,
            "vram_delta_mib":  vram_delta,
            "gen_tps":    gen_tps,
            "response":   result,
        })

    try:
        server.unload_all_models(provider)
    except Exception:
        pass

## Other Providers: GPU Auto-Managed

Ollama, LM Studio, and Unsloth control GPU placement internally. We load once and verify the model lands on GPU (significant VRAM increase).

In [ ]:
other_results = []
messages = [{"role": "user", "content": PROMPT}]

for provider in ["ollama", "lm_studio", "unsloth"]:
    if not statuses.get(provider, {}).get("ok"):
        print(f"{provider} SKIP")
        continue
    model = PREFERRED_MODELS.get(provider)
    if not model:
        continue

    print(f"\n--- {provider} / {model} ---")

    try:
        server.unload_all_models(provider)
        time.sleep(2)
    except Exception:
        pass

    vram_before = vram_used_mib()

    try:
        llm = server.load_model(provider, model)
        t0 = time.perf_counter()
        result, usage = await llm.call(
            return_usage=True,
            messages=messages,
            temperature=0.0,
            options=OPTIONS,
        )
        elapsed = time.perf_counter() - t0
    except Exception as exc:
        print(f"  ERROR: {exc}")
        other_results.append({"provider": provider, "error": str(exc)})
        continue

    vram_after = vram_used_mib()
    vram_delta = vram_after - vram_before
    gen_tok = usage.get("completion_tokens", 0)
    gen_tps = round(gen_tok / elapsed, 1) if elapsed > 0 else 0

    on_gpu = "YES" if vram_delta > 1000 else "NO"
    print(f"  VRAM before : {vram_before} MiB")
    print(f"  VRAM after  : {vram_after} MiB")
    print(f"  VRAM delta  : {vram_delta} MiB  → on GPU: {on_gpu}")
    print(f"  Speed       : {gen_tps} tok/s")
    print(f"  Reply       : {result!r}")

    other_results.append({
        "provider":       provider,
        "vram_before_mib": vram_before,
        "vram_after_mib":  vram_after,
        "vram_delta_mib":  vram_delta,
        "on_gpu":          on_gpu,
        "gen_tps":         gen_tps,
        "response":        result,
    })

    try:
        server.unload_all_models(provider)
        time.sleep(1)
    except Exception:
        pass

## Summary

In [ ]:
print("=== llama_cpp gpu_layers test ===")
print(f"{'Config':<20}  {'VRAM delta':>12}  {'Speed':>10}  {'Pass?'}")
print("-" * 55)
for r in llama_results:
    if "error" in r:
        print(f"{r['label']:<20}  ERROR: {r['error'][:30]}")
        continue
    expected_gpu = r["gpu_layers"] > 0
    got_gpu = r["vram_delta_mib"] > 1000
    passed = "PASS" if expected_gpu == got_gpu else "FAIL"
    print(f"{r['label']:<20}  {r['vram_delta_mib']:>10} MiB  {r['gen_tps']:>8.1f} t/s  {passed}")

print()
print("=== other providers (auto-managed GPU) ===")
print(f"{'Provider':<12}  {'VRAM delta':>12}  {'Speed':>10}  {'On GPU?'}")
print("-" * 52)
for r in other_results:
    if "error" in r:
        print(f"{r['provider']:<12}  ERROR: {r['error'][:40]}")
        continue
    print(f"{r['provider']:<12}  {r['vram_delta_mib']:>10} MiB  {r['gen_tps']:>8.1f} t/s  {r['on_gpu']}")